# 🌊 Sonaris — Train DRISHTI-SSS Sonar Dataset (5 Classes)
### Automated Training on Google Colab (T4 GPU)

This notebook trains **YOLO11 Nano** on the **`drishti_clean`** sonar dataset shared on Google Drive:
- `0: crab_pot`
- `1: submarine_pipeline` (1,321 objects)
- `2: shipwreck` (2,623 objects)
- `3: ghost_net` (1,140 objects)
- `4: mine_cylinder` (1,018 objects)

⏱️ **Estimated Training Time**: ~25-30 minutes on free T4 GPU.

## Step 1: Check GPU & Install Dependencies

In [ ]:
!pip install ultralytics -q

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Please enable GPU: Runtime -> Change runtime type -> T4 GPU")

## Step 2: Mount Google Drive
> **IMPORTANT**: Before running this, make sure you clicked **"Add shortcut to Drive"** on the shared folder link ([https://drive.google.com/drive/folders/1QvzLWu4t13Ng2LOOpI71FtnbJa3YHnEp](https://drive.google.com/drive/folders/1QvzLWu4t13Ng2LOOpI71FtnbJa3YHnEp)) so `drishti_clean` appears in your Google Drive!

In [ ]:
from google.colab import drive
import os
import glob

drive.mount('/content/drive')

# Search for drishti_clean folder in your Google Drive
candidates = glob.glob('/content/drive/MyDrive/**/drishti_clean', recursive=True)
if not candidates:
    candidates = glob.glob('/content/drive/MyDrive/**/drishti*', recursive=True)

if candidates:
    drishti_path = candidates[0]
    print(f"✅ Found DRISHTI dataset at: {drishti_path}")
    print("Folder contents:", os.listdir(drishti_path))
else:
    print("❌ 'drishti_clean' not found in your Drive.")
    print("Please open the shared link -> click 'Add shortcut to Drive' -> select 'My Drive'")

## Step 3: Fast-Copy to Colab SSD & Create YAML Config
*(Copying to local `/content/dataset_drishti` speeds up training by 10x compared to reading over Google Drive)*


In [ ]:
import shutil

# Copy to local fast SSD
local_dataset = "/content/dataset_drishti"
if not os.path.exists(local_dataset):
    print("Copying dataset to Colab local SSD for high-speed training...")
    shutil.copytree(drishti_path, local_dataset, dirs_exist_ok=True, ignore=shutil.ignore_patterns('*.tmp', '.git*'))
    print("✅ Copy complete!")
else:
    print("✅ Dataset already copied to local SSD.")

# Create the exact YAML configuration matching DRISHTI structure
yaml_content = """
path: /content/dataset_drishti
train: train/images
val: val/images
test: test/images

task: detect
nc: 5
names:
  0: crab_pot
  1: submarine_pipeline
  2: shipwreck
  3: ghost_net
  4: mine_cylinder
"""

with open('/content/drishti_colab.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("✅ Created /content/drishti_colab.yaml with all 5 classes!")


## Step 4: Train YOLO11 Nano on the 5 DRISHTI Classes

In [ ]:
from ultralytics import YOLO

# Load YOLO11 nano model
model = YOLO('yolo11n.pt')

# Train with optimal acoustic sonar settings
results = model.train(
    data='/content/drishti_colab.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,
    device=0 if torch.cuda.is_available() else 'cpu',
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.001,
    cos_lr=True,
    augment=True,
    mosaic=0.5,
    project='/content/drive/MyDrive/sonaris_runs',  # Automatically saves to your Google Drive!
    name='drishti_yolo11',
    exist_ok=True
)

print("\n🎉 Training Complete!")
print("Model saved to your Google Drive at: sonaris_runs/drishti_yolo11/weights/best.pt")

## Step 5: Evaluate & Download Weights

In [ ]:
from google.colab import files
import os

# Evaluate on test split
metrics = model.val(data='/content/drishti_colab.yaml', split='test')
print(f"\nOverall Test mAP@50: {metrics.box.map50:.3f}")

# Download best.pt directly
best_weights = '/content/drive/MyDrive/sonaris_runs/drishti_yolo11/weights/best.pt'
if os.path.exists(best_weights):
    print(f"Downloading {best_weights}...")
    files.download(best_weights)
else:
    print("Searching for local weights...")
    import glob
    local_weights = glob.glob('**/best.pt', recursive=True)
    if local_weights:
        files.download(max(local_weights, key=os.path.getmtime))